# Thiếu văn minh: PhoBERT

Fine-tune `vinai/phobert-base-v2` trên UIT-ViCTSD với ba seed (42, 43, 44).
Tách từ bằng PyVi, dùng văn bản của bình luận và lấy trung bình xác suất lớp 1 từ ba model.

Sau khi chốt nhãn dev, chạy `python -m src.classifiers.incivility kaggle-bundle`.
Gắn bundle dưới dạng Kaggle dataset private. Bật Internet và GPU T4, dùng PyTorch 2.6 trở lên.
Nếu chạy trên GPU cục bộ, đường dẫn mặc định là `data/kaggle/incivility`.

Mặc định chạy `dev`. Nếu chọn PhoBERT làm model cuối, đặt `VISIM_INCIVILITY_PHASE=corpus`
trong môi trường hoặc trong cell đầu trước khi chạy. Notebook sẽ huấn luyện lại và lưu
ba checkpoint `.safetensors` cùng `corpus_scores_phobert_victsd_comment.csv`.
Chép các file scores và JSON từ `/kaggle/working` vào `results/incivility/kaggle/`.

In [ ]:
%pip install -q transformers==4.57.1 pyvi==0.1.1

In [ ]:
import hashlib
import json
import os
import platform
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PHASE = os.environ.get("VISIM_INCIVILITY_PHASE", "dev")
if PHASE not in ("dev", "corpus"):
    raise ValueError("VISIM_INCIVILITY_PHASE must be dev or corpus")

kaggle_input = Path("/kaggle/input")
if "VISIM_INCIVILITY_DATA" in os.environ:
    data_dir = Path(os.environ["VISIM_INCIVILITY_DATA"])
elif kaggle_input.is_dir():
    manifests = sorted(kaggle_input.rglob("incivility_manifest.json"))
    if len(manifests) != 1:
        raise ValueError("Attach exactly one VISIM incivility dataset")
    data_dir = manifests[0].parent
else:
    data_dir = Path("data/kaggle/incivility")

out_dir = Path(os.environ.get(
    "VISIM_INCIVILITY_OUTPUT",
    "/kaggle/working" if kaggle_input.is_dir() else "results/incivility/kaggle",
))
out_dir.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = json.loads((data_dir / "incivility_manifest.json").read_text(encoding="utf-8"))


def read_data(name):
    path = data_dir / f"incivility_{name}.csv"
    hash_key = "bundle_train_sha256" if name == "train" else f"{name}_sha256"
    if sha256(path) != manifest[hash_key]:
        raise ValueError(f"{path.name} does not match the input manifest")
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if frame["id"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate IDs")
    return frame


def write_scores(frame, score, config):
    if len(score) != len(frame) or not (np.isfinite(score).all() and ((0 <= score) & (score <= 1)).all()):
        raise ValueError(f"invalid scores for {config}")
    path = out_dir / f"{PHASE}_scores_{config}.csv"
    temporary = path.with_suffix(".csv.tmp")
    pd.DataFrame({
        "id": frame["id"], "pred": (score >= 0.5).astype(int), "score": score,
    }).to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)
    return path


def write_run(family, configs, files, **settings):
    report = {
        "phase": PHASE, "data": manifest, "configs": configs,
        "output_sha256": {p.name: sha256(p) for p in files},
        "where": "Kaggle" if kaggle_input.is_dir() else "local GPU",
        "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
        "python": platform.python_version(), "torch": str(torch.__version__),
        "transformers": version("transformers"), **settings,
    }
    path = out_dir / f"{family}_{PHASE}.json"
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)
    print(path)


if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU before running this notebook.")
device = torch.device("cuda:0")
print("Data:", data_dir, "| Output:", out_dir, "| GPU:", torch.cuda.get_device_name(device))

In [ ]:
import random

from packaging.version import Version
from pyvi import ViTokenizer
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

if Version(torch.__version__.split("+")[0]) < Version("2.6"):
    raise RuntimeError("PhoBERT's .bin weights require PyTorch 2.6 or newer with this Transformers version.")

MODEL = "vinai/phobert-base-v2"
REVISION = "86cd7fd4c148980922ac11a2cf5e257f2ba639e1"
CONFIG = "phobert_victsd_comment"
SEEDS = (42, 43, 44)
EPOCHS, LR, BATCH, MAX_LENGTH = 3, 2e-5, 32, 192
WEIGHT_DECAY, WARMUP_FRACTION = 0.01, 0.1

tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, use_fast=False)


def encode(frame, training=False):
    sentences = [ViTokenizer.tokenize(text) for text in frame["text"]]
    tokens = tok(sentences, truncation=True, max_length=MAX_LENGTH)
    items = [{"input_ids": ids, "attention_mask": mask}
             for ids, mask in zip(tokens["input_ids"], tokens["attention_mask"])]
    if training:
        for item, label in zip(items, frame["label"]):
            item["labels"] = int(label)
    return items


def collate(items):
    return tok.pad(items, padding=True, pad_to_multiple_of=8, return_tensors="pt")


def train_and_score(seed, train_items, target_loader, weights):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    train_loader = DataLoader(train_items, batch_size=BATCH, shuffle=True, collate_fn=collate,
                              generator=torch.Generator().manual_seed(seed))
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, revision=REVISION, num_labels=2, use_safetensors=False,
        id2label={0: "civil", 1: "uncivil"}, label2id={"civil": 0, "uncivil": 1},
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    schedule = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    for epoch in range(EPOCHS):
        for batch in train_loader:
            batch = {key: tensor.to(device) for key, tensor in batch.items()}
            y = batch.pop("labels")
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(model(**batch).logits, y, weight=weights)
            scaler.scale(loss).backward()
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                schedule.step()
        print(f"Seed {seed}, epoch {epoch + 1}/{EPOCHS}", flush=True)

    model.eval()
    scores = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        for batch in target_loader:
            batch = {key: tensor.to(device) for key, tensor in batch.items()}
            logits = model(**batch).logits
            scores.append(logits.float().softmax(-1)[:, 1].cpu().numpy())
    return model, np.concatenate(scores)

In [ ]:
started = time.perf_counter()
train, target = read_data("train"), read_data(PHASE)
train_items = encode(train, training=True)
target_loader = DataLoader(encode(target), batch_size=BATCH * 2, collate_fn=collate)
counts = np.bincount(train["label"].to_numpy(dtype=int), minlength=2)
weights = torch.as_tensor(len(train) / (2 * counts), dtype=torch.float32, device=device)
scores, files, members = [], [], []
for seed in SEEDS:
    start = time.perf_counter()
    model, score = train_and_score(seed, train_items, target_loader, weights)
    scores.append(score)
    if PHASE == "dev":
        files.append(write_scores(target, score, f"{CONFIG}_seed_{seed}"))
    else:
        checkpoint = out_dir / f"{CONFIG}_seed_{seed}"
        model.save_pretrained(checkpoint, safe_serialization=True)
        tok.save_pretrained(checkpoint)
    members.append({"seed": seed, "seconds": round(time.perf_counter() - start, 2)})
    del model
    torch.cuda.empty_cache()

files.append(write_scores(target, np.mean(scores, axis=0), CONFIG))
write_run(
    "phobert", {CONFIG: {"seconds": round(time.perf_counter() - started, 2), "members": members}}, files,
    model=MODEL, revision=REVISION, seeds=SEEDS, epochs=EPOCHS, lr=LR, batch=BATCH, max_length=MAX_LENGTH,
    weight_decay=WEIGHT_DECAY, warmup_fraction=WARMUP_FRACTION, class_weights=weights.cpu().tolist(),
    preprocessing=f"PyVi {version('pyvi')}", train_n=len(train),
)